# 🌿 AgroDetect AI – Plant Disease Classification Engine
> Transfer Learning with MobileNetV2 | TensorFlow/Keras | PlantVillage Dataset


## 📦 Step 0 – Install Dependencies


In [ ]:
# Run ONCE to install all required packages
!pip install -r requirements.txt

## 📁 Step 1 – Prepare Dataset


In [ ]:
# Option A: Download from Kaggle (requires kaggle.json)
!python src/prepare_data.py --kaggle

# Option B: Extract local zip
# !python src/prepare_data.py --zip data/raw/plantvillage.zip

## 🏗️ Step 2 – Build & Train Model


In [ ]:
import os, json, numpy as np, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 20
DATA_DIR   = 'data/processed'

datagen = ImageDataGenerator(rescale=1./255, rotation_range=20,
    width_shift_range=0.2, height_shift_range=0.2,
    shear_range=0.2, zoom_range=0.2,
    horizontal_flip=True, validation_split=0.2)

train_gen = datagen.flow_from_directory(DATA_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', subset='training')
val_gen   = datagen.flow_from_directory(DATA_DIR, target_size=IMG_SIZE,
    batch_size=BATCH_SIZE, class_mode='categorical', subset='validation')

NUM_CLASSES = len(train_gen.class_indices)
print(f'Detected {NUM_CLASSES} disease classes')

os.makedirs('outputs', exist_ok=True)
with open('outputs/class_labels.json', 'w') as f:
    json.dump({str(v): k for k, v in train_gen.class_indices.items()}, f, indent=2)

In [ ]:
base = MobileNetV2(input_shape=(*IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
out = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(base.input, out)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
os.makedirs('models', exist_ok=True)
callbacks = [
    ModelCheckpoint('models/agrodetect_mobilenet.h5', save_best_only=True, monitor='val_accuracy', verbose=1),
    EarlyStopping(patience=5, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.2, patience=3, verbose=1)
]
history = model.fit(train_gen, epochs=EPOCHS, validation_data=val_gen, callbacks=callbacks)

## 📈 Step 3 – Plot Training Curves


In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
a1.plot(history.history['accuracy'],     label='Train', color='#2ecc71')
a1.plot(history.history['val_accuracy'], label='Val',   color='#27ae60', linestyle='--')
a1.set_title('Accuracy'); a1.legend()
a2.plot(history.history['loss'],     label='Train', color='#e74c3c')
a2.plot(history.history['val_loss'], label='Val',   color='#c0392b', linestyle='--')
a2.set_title('Loss'); a2.legend()
plt.tight_layout()
plt.savefig('outputs/training_curves.png', dpi=150)
plt.show()

## 🔍 Step 4 – Predict on a Single Image


In [ ]:
import cv2

IMAGE_PATH = 'data/raw/sample_leaf.jpg'   # ← change this

with open('outputs/class_labels.json') as f:
    labels = json.load(f)

saved_model = tf.keras.models.load_model('models/agrodetect_mobilenet.h5')

img   = cv2.resize(cv2.cvtColor(cv2.imread(IMAGE_PATH), cv2.COLOR_BGR2RGB), (224, 224))
inp   = np.expand_dims(img / 255.0, 0)
preds = saved_model.predict(inp)[0]
top5  = np.argsort(preds)[::-1][:5]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(img); ax1.set_title('Input Leaf'); ax1.axis('off')
top_names = [labels[str(i)].replace('_', ' ') for i in top5]
top_probs  = [preds[i] for i in top5]
ax2.barh(top_names[::-1], top_probs[::-1], color='#2ecc71')
ax2.set_xlabel('Confidence'); ax2.set_title('Top-5 Predictions')
plt.tight_layout(); plt.show()
print(f"\n🌿 Prediction: {top_names[0]}  ({top_probs[0]*100:.1f}%)")

## 📊 Step 5 – Evaluate & Confusion Matrix


In [ ]:
!python src/evaluate.py